In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
llm = ChatOllama(model="llama3.2")


In [7]:
class JokeState(TypedDict): 
    topic: str 
    joke: str 
    explanation: str

In [8]:
def generate_joke(state: JokeState): 
    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content


    return{'joke': response}

In [9]:
def generate_explanation(state: JokeState): 
    prompt = f"Write an explanation for the given joke = {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

#node
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

#edges 
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)


checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config= {"configurable": {"thread_id": "1"}}
inital_state = {
    "topic": "Politics"
}

workflow.invoke(inital_state, config=config)

{'topic': 'Politics',
 'joke': 'Why did the politician bring a ladder to the debate?\n\nBecause he wanted to take his argument to a higher level!',
 'explanation': 'This joke is a play on words, using a common idiomatic expression to create a pun. The phrase "take it to a higher level" is often used to describe someone who is trying to elevate their argument or discussion to a more mature or sophisticated level.\n\nIn this joke, the politician brings a ladder to the debate, which is a literal device used to elevate or raise something. However, the phrase "take it to a higher level" also implies that the politician is trying to improve or improve upon their argument. The joke relies on this dual meaning of the phrase, playing on the idea that the politician is both physically and metaphorically raising their argument to a higher level.\n\nThe joke is a clever play on words, using a common expression to create a clever and unexpected punchline. It requires a quick mental shift to underst

In [12]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'Politics', 'joke': 'Why did the politician bring a ladder to the debate?\n\nBecause he wanted to take his argument to a higher level!', 'explanation': 'This joke is a play on words, using a common idiomatic expression to create a pun. The phrase "take it to a higher level" is often used to describe someone who is trying to elevate their argument or discussion to a more mature or sophisticated level.\n\nIn this joke, the politician brings a ladder to the debate, which is a literal device used to elevate or raise something. However, the phrase "take it to a higher level" also implies that the politician is trying to improve or improve upon their argument. The joke relies on this dual meaning of the phrase, playing on the idea that the politician is both physically and metaphorically raising their argument to a higher level.\n\nThe joke is a clever play on words, using a common expression to create a clever and unexpected punchline. It requires a quick ment